<a href="https://colab.research.google.com/github/czechuuu/micro-vla/blob/main/MicroVla.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

MIMUW Reinforcement Learning 26/26L

Adrian Boguszewski, Bartosz Czechowski

# Project statement
## Overview & Motivation
Vision-Language-Action (VLA) models, such as RT-2 and OpenVLA, have recently demonstrated incredible zero-shot generalization in robotics. However, these models face a severe data bottleneck: while we have massive repositories of video data showing humans or robots completing tasks, we have very little action-annotated data containing the exact motor torques and joint commands required to replicate those movements.
Standard supervised approaches fail when actions are missing. If we can design architectures that learn world dynamics and physics directly from "observation-only" videos, we can vastly expand the training sets for robotic foundation models. This project explores whether pre-training on a large corpus of action-free manipulation videos can improve a small-scale VLA's ability to perform In-Context Learning (ICL) from just one or two fully annotated demonstrations.
## Project Objective
The primary goal of this project is to build a "Micro-VLA" agent that improves its sample efficiency and few-shot generalization by learning from unannotated video trajectories. The student will implement a system that uses an Inverse Dynamics Model (or masked token prediction) to ingest observation-only sequences from a simulated robotics environment, combining this with a small set of action-annotated data to solve continuous control manipulation tasks.
## Expected Deliverables (Pass Criteria)
To successfully pass this project, students are expected to complete the following concrete tasks:
- **Simulated Benchmark Setup**: Set up a lightweight, continuous-control robotic manipulation environment (e.g., Robomimic or Meta-World). Generate a synthetic dataset: a small fraction containing full state-action-reward data, and a large fraction stripped of actions to simulate "video-only" observations.
- **Architecture Implementation**: Design a small Transformer-based policy (e.g., a miniaturized Decision Transformer). Implement an auxiliary objective, such as an Inverse Dynamics module, that forces the network to predict the missing actions between two consecutive video frames, allowing it to learn from the observation-only dataset.
- **Benchmarking & Evaluation**: Train two models: a baseline Micro-VLA trained strictly on the small, fully-annotated dataset, and your proposed Micro-VLA trained on the mixed dataset.
- **Comparative Analysis**: Deliver a final report evaluating the models on their In-Context Learning capabilities. Specifically, prompt the frozen models with a single successful demonstration of a new, unseen task variation (e.g., picking up a differently colored object) and measure which model exhibits better zero-shot or few-shot transfer.


# 1. Simulated Benchmark Setup


## Downloading dependencies

In [1]:
import os
os.environ["MUJOCO_GL"] = "egl"

In [2]:
# due to conflicts with other packages
!pip uninstall flax jax jaxlib -y
!pip install diffusers==0.11.1

Found existing installation: flax 0.11.2
Uninstalling flax-0.11.2:
  Successfully uninstalled flax-0.11.2
Found existing installation: jax 0.7.2
Uninstalling jax-0.7.2:
  Successfully uninstalled jax-0.7.2
Found existing installation: jaxlib 0.7.2
Uninstalling jaxlib-0.7.2:
  Successfully uninstalled jaxlib-0.7.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 524.9/524.9 kB 17.1 MB/s eta 0:00:00
  Attempting uninstall: diffusers
    Found existing installation: diffusers 0.37.1
    Uninstalling diffusers-0.37.1:
      Successfully uninstalled diffusers-0.37.1


In [3]:
# install robomimic via git
!git clone https://github.com/ARISE-Initiative/robomimic
!pip install -e robomimic/

Cloning into 'robomimic'...
remote: Enumerating objects: 3572, done.
remote: Total 3572 (delta 0), reused 0 (delta 0), pack-reused 3572 (from 1)
Receiving objects: 100% (3572/3572), 62.21 MiB | 20.98 MiB/s, done.
Resolving deltas: 100% (2432/2432), done.
Obtaining file:///content/robomimic
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.5/217.5 kB 9.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 402.6/402.6 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 97.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 96.3 MB/s eta 0:00:00
  Created wheel for egl_probe: filename=egl_probe-1.0.2-cp312-cp312-linux_x86_64.whl size=478088 sha256=b84ae16a3dfbde6e24801c704d43cd4ff315ef9cef410583

In [4]:
# an mujoco and robosuite normally
# rstarts the session due to some numpy mismath
!sudo apt install curl git libgl1-mesa-dev libgl1-mesa-glx libglew-dev \
         libosmesa6-dev software-properties-common net-tools unzip vim \
         virtualenv wget xpra xserver-xorg-dev libglfw3-dev patchelf < echo "31 \n 1"

!pip install mujoco
!pip install robosuite

/bin/bash: line 1: echo: No such file or directory
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.5/42.5 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 70.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.5/243.5 kB 11.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 397.9 kB/s eta 0:00:00
INFO: pip is looking at multiple versions of opencv-python to determine which version is compatible with other requirements. This could take a while.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.8/156.8 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 896.8/896.8 kB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 79.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.8/106.8 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━

## Downloading the raw dataset

In [1]:
! python /content/robomimic/robomimic/scripts/download_datasets.py --tasks lift --dataset_types ph --hdf5_types raw

2026-05-15 11:34:52.795486: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
config.json: 4.52kB [00:00, 16.5MB/s]
model.safetensors: 100% 1.71G/1.71G [00:08<00:00, 210MB/s]
tokenizer_config.json: 100% 905/905 [00:00<00:00, 5.37MB/s]
vocab.json: 961kB [00:00, 36.5MB/s]
merges.txt: 525kB [00:00, 113MB/s]
tokenizer.json: 2.22MB [00:00, 161MB/s]
special_tokens_map.json: 100% 389/389 [00:00<00:00, 3.47MB/s]
ROBOMIMIC WARNING(
    No private macro file found!
    It is recommended to use a private macro file
    To setup, run: python /content/robomimic/robomimic/scripts/setup_macros.py
)

    task: lift
    dataset type: ph
    hdf5 type: raw
    download path: /content/robomimic/datasets/lift/ph
demo_v15.hdf5: 100% 36.7M/36.7M [00:00<00:00, 108MB/s] 


In [3]:
import os
download_folder = "/content/robomimic/datasets/lift/ph"
dataset_path = os.path.join(download_folder, "demo_v15.hdf5")
assert os.path.exists(dataset_path)

## Postprocessing the raw dataset

There is the `robomimic/scripts/dataset_states_to_obs.py` script which should do the same thing OOTB but I couldn't get it to work with the colab GPU and rendering the images would crash. So this code does +- the same thing but works on colab.

In [4]:
import os
# CRITICAL: Must be set before any MuJoCo imports
os.environ['MUJOCO_GL'] = 'egl'

import h5py
import json
import numpy as np
import robosuite as suite
from tqdm import tqdm

# ==========================================
# CONFIGURATION
# ==========================================
INPUT_PATH = "/content/robomimic/datasets/lift/ph/demo_v15.hdf5"
OUTPUT_PATH = "./full_dataset.hdf5"
CAMERAS = ["agentview", "robot0_eye_in_hand"]
RESOLUTION = 224
COMPRESSION_LEVEL = 4 # 0-9

# ==========================================
# HELPER FUNCTIONS
# ==========================================
def sanitize_env_kwargs(kwargs):
    """Injects our VLA camera requirements into the legacy metadata."""
    for key in ["camera_height", "camera_width", "camera_name"]:
        kwargs.pop(key, None)

    kwargs.update({
        "has_renderer": False,
        "has_offscreen_renderer": True,
        "use_camera_obs": True,
        "camera_names": CAMERAS,
        "camera_heights": RESOLUTION,
        "camera_widths": RESOLUTION
    })
    return kwargs

# ==========================================
# MAIN EXTRACTION LOOP
# ==========================================
def generate_dataset():
    print(f"Opening {INPUT_PATH}...")

    with h5py.File(INPUT_PATH, "r") as f_in, h5py.File(OUTPUT_PATH, "w") as f_out:

        # 1. Copy over metadata
        data_grp_out = f_out.create_group("data")
        data_grp_out.attrs["env_args"] = f_in["data"].attrs["env_args"]

        # 2. Initialize Environment
        env_meta = json.loads(f_in["data"].attrs["env_args"])
        kwargs = sanitize_env_kwargs(env_meta["env_kwargs"])

        print(f"Initializing pure Robosuite environment: {env_meta['env_name']}...")
        env = suite.make(env_meta["env_name"], **kwargs)

        # 3. AUTO-DETECT EVERYTHING
        dummy_obs = env.reset()

        # Sort keys by shape: 3D arrays are images, 1D arrays are states
        image_keys = [k for k, v in dummy_obs.items() if isinstance(v, np.ndarray) and len(v.shape) >= 3]
        state_keys = [k for k, v in dummy_obs.items() if isinstance(v, np.ndarray) and len(v.shape) == 1]

        print(f"Omni-Capture Mode Engaged:")
        print(f"📸 Detected {len(image_keys)} Image streams: {image_keys}")
        print(f"📊 Detected {len(state_keys)} State streams.")

        # 4. Process every demonstration
        demo_keys = list(f_in["data"].keys())[:2]

        for demo_key in tqdm(demo_keys, desc="Rendering Trajectories"):
            demo_in = f_in[f"data/{demo_key}"]
            actions = demo_in["actions"][:]
            states = demo_in["states"][:]
            num_steps = len(actions)

            # Setup output groups
            demo_out = data_grp_out.create_group(demo_key)
            demo_out.attrs["num_samples"] = num_steps
            demo_out.create_dataset("actions", data=actions)
            demo_out.create_dataset("states", data=states)
            obs_out = demo_out.create_group("obs")

            # Pre-allocate master buffers
            buffers = {}
            for k in image_keys:
                # E.g., (T, 224, 224, 3)
                buffers[k] = np.empty((num_steps, *dummy_obs[k].shape), dtype=np.uint8)
            for k in state_keys:
                # E.g., (T, 7) or (T, 3)
                buffers[k] = np.empty((num_steps, dummy_obs[k].shape[0]), dtype=np.float32)

            # Reset physics
            env.reset()
            env.sim.set_state_from_flattened(states[0])
            env.sim.forward()

            # Replay and record
            for t, action in enumerate(actions):
                obs, _, _, _ = env.step(action)

                # Capture all images (and flip them)
                for k in image_keys:
                    buffers[k][t] = np.flipud(obs[k])

                # Capture all states
                for k in state_keys:
                    buffers[k][t] = obs[k]

            # Save to disk
            for k in image_keys:
                obs_out.create_dataset(
                    k, data=buffers[k], dtype=np.uint8,
                    compression="gzip", compression_opts=COMPRESSION_LEVEL
                )

            for k in state_keys:
                obs_out.create_dataset(k, data=buffers[k], dtype=np.float32)

    print(f"\n✅ Omni-Dataset successfully compiled to: {OUTPUT_PATH}")


[robosuite WARNING] No private macro file found! (macros.py:57)
[robosuite WARNING] It is recommended to use a private macro file (macros.py:58)
[robosuite WARNING] To setup, run: python /usr/local/lib/python3.12/dist-packages/robosuite/scripts/setup_macros.py (macros.py:59)
[robosuite WARNING] Could not import robosuite_models. Some robots may not be available. If you want to use these robots, please install robosuite_models from source (https://github.com/ARISE-Initiative/robosuite_models) or through pip install. (__init__.py:30)
[robosuite WARNING] Could not load the mink-based whole-body IK. Make sure you install related import properly, otherwise you will not be able to use the default IK controller setting for GR1 robot. (__init__.py:40)


In [5]:
generate_dataset()

Opening /content/robomimic/datasets/lift/ph/demo_v15.hdf5...
Initializing pure Robosuite environment: Lift...
Omni-Capture Mode Engaged:
📸 Detected 2 Image streams: ['agentview_image', 'robot0_eye_in_hand_image']
📊 Detected 15 State streams.


Rendering Trajectories: 100%|██████████| 2/2 [00:28<00:00, 14.43s/it]


✅ Omni-Dataset successfully compiled to: ./full_dataset.hdf5


# 2137. Creating PuckStack

In [1]:
! pip install robosuite

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.5/42.5 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.7 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of opencv-python to determine which version is compatible with other requirements. This could take a while.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.8/156.8 MB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 896.8/896.8 kB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 128.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 97.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.8/106.8 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.0/92.0 kB 10.3 MB/s eta 0:00:0

In [1]:
! pip install imageio imageio-ffmpeg

In [2]:
import os
# CRITICAL: This must be set BEFORE importing robosuite or mujoco for headless rendering.
# Use 'egl' for GPU instances (like Colab) or 'osmesa'/'glfw' for CPU-only.
os.environ['MUJOCO_GL'] = 'egl'

import numpy as np
import imageio
import robosuite as suite
from robosuite.models.arenas import TableArena
from robosuite.environments.manipulation.stack import Stack
from robosuite.models.objects import CylinderObject
from robosuite.models.tasks import ManipulationTask
from robosuite.utils.placement_samplers import UniformRandomSampler

class PuckStack(Stack):
    """
    A custom environment that inherits from the standard Stack task,
    but replaces the square boxes with flat cylindrical pucks.
    """
    def _load_model(self):
        """
        We override the model loading phase to inject our custom objects
        instead of the default BoxObjects.
        """
        # Call the parent of Stack (ManipulationEnv) to load the robot and table arena
        super(Stack, self)._load_model()

        xpos = self.robots[0].robot_model.base_xpos_offset["table"](self.table_full_size[0])
        self.robots[0].robot_model.set_base_xpos(xpos)

        mujoco_arena = TableArena(
            table_full_size=self.table_full_size,
            table_friction=self.table_friction,
            table_offset=self.table_offset,
        )

        # Arena always gets set to zero origin
        mujoco_arena.set_origin([0, 0, 0])

        # 1. Define the Pucks (Cylinders)
        # We keep the names "cubeA" and "cubeB" so the base class reward function
        # and observation setup still work perfectly without modification!
        self.cubeA = CylinderObject(
            name="cubeA",
            size=[0.04, 0.02],          # [Radius, Half-height]
            rgba=[1, 0, 0, 1],          # Red
            friction=[1.0, 0.005, 0.0001] # Increased friction for grip
        )
        self.cubeB = CylinderObject(
            name="cubeB",
            size=[0.04, 0.02],
            rgba=[0, 1, 0, 1],          # Green
            friction=[1.0, 0.005, 0.0001]
        )

        # 2. Set up the object placement sampler (where they spawn on the table)
        self.placement_initializer = UniformRandomSampler(
            name="ObjectSampler",
            mujoco_objects=[self.cubeA, self.cubeB],
            x_range=[-0.1, 0.1],
            y_range=[-0.1, 0.1],
            rotation=None, # Random rotations
            ensure_object_boundary_in_range=False,
            ensure_valid_placement=True,
            reference_pos=self.table_offset,
            z_offset=0.01,
        )

        # 3. Build the actual MuJoCo task containing the arena, robots, and our pucks
        self.model = ManipulationTask(
            mujoco_arena=mujoco_arena,
            mujoco_robots=[robot.robot_model for robot in self.robots],
            mujoco_objects=[self.cubeA, self.cubeB],
        )


# Initialize our custom PuckStack environment for Headless Video Recording
env = PuckStack(
    robots="Panda",             # Using the standard Franka Emika Panda
    has_renderer=False,         # Must be False in headless environments
    has_offscreen_renderer=True,# Enable background rendering for video
    use_camera_obs=True,        # Must be True to capture pixel arrays
    camera_names="frontview",   # The camera angle to record
    control_freq=20,            # 20Hz control frequency
    horizon=200,                # Max steps per episode (shortened for a quick 10s video)
)

# Reset to start the simulation
obs = env.reset()
frames = []

print("Running headless PuckStack Environment...")
print("Recording random actions to memory...")

# Run a random action loop
for step in range(env.horizon):
    # Generate a random action within the valid action space
    action = np.random.uniform(-1, 1, env.action_dim)

    # Step the environment
    obs, reward, done, info = env.step(action)
    # Grab the image frame from the observation dictionary
    # MuJoCo images render upside down, so we flip it vertically
    frame = obs["frontview_image"]
    frame = np.flipud(frame)
    frames.append(frame)

print("Saving video to puck_stack.mp4...")

# Save the captured frames into an mp4 video file
writer = imageio.get_writer('puck_stack.mp4', fps=env.control_freq)
for frame in frames:
    writer.append_data(frame)
writer.close()

print("Video saved successfully! You can now download or view puck_stack.mp4")

[robosuite WARNING] No private macro file found! (macros.py:57)
[robosuite WARNING] It is recommended to use a private macro file (macros.py:58)
[robosuite WARNING] To setup, run: python /usr/local/lib/python3.12/dist-packages/robosuite/scripts/setup_macros.py (macros.py:59)
[robosuite WARNING] Could not import robosuite_models. Some robots may not be available. If you want to use these robots, please install robosuite_models from source (https://github.com/ARISE-Initiative/robosuite_models) or through pip install. (__init__.py:30)
[robosuite WARNING] Could not load the mink-based whole-body IK. Make sure you install related import properly, otherwise you will not be able to use the default IK controller setting for GR1 robot. (__init__.py:40)
[robosuite INFO] Loading controller configuration from: /usr/local/lib/python3.12/dist-packages/robosuite/controllers/config/robots/default_panda.json (composite_controller_factory.py:121)
INFO:robosuite_logs:Loading controller configuration from

Running headless PuckStack Environment...
Recording random actions to memory...
Saving video to puck_stack.mp4...
Video saved successfully! You can now download or view puck_stack.mp4


In [4]:
import os
# CRITICAL: This must be set BEFORE importing robosuite or mujoco for headless rendering.
# Use 'egl' for GPU instances (like Colab) or 'osmesa'/'glfw' for CPU-only.
os.environ['MUJOCO_GL'] = 'egl'

import numpy as np
import imageio
import robosuite as suite
from robosuite.environments.manipulation.stack import Stack

# Initialize our custom PuckStack environment for Headless Video Recording
env = Stack(
    robots="Panda",             # Using the standard Franka Emika Panda
    has_renderer=False,         # Must be False in headless environments
    has_offscreen_renderer=True,# Enable background rendering for video
    use_camera_obs=True,        # Must be True to capture pixel arrays
    camera_names="frontview",   # The camera angle to record
    control_freq=20,            # 20Hz control frequency
    horizon=200,                # Max steps per episode (shortened for a quick 10s video)
)

# Reset to start the simulation
obs = env.reset()
frames = []

print("Running headless PuckStack Environment...")
print("Recording random actions to memory...")

# Run a random action loop
for step in range(env.horizon):
    # Generate a random action within the valid action space
    action = np.random.uniform(-1, 1, env.action_dim)

    # Step the environment
    obs, reward, done, info = env.step(action)
    # Grab the image frame from the observation dictionary
    # MuJoCo images render upside down, so we flip it vertically
    frame = obs["frontview_image"]
    frame = np.flipud(frame)
    frames.append(frame)

print("Saving video to puck_stack.mp4...")

# Save the captured frames into an mp4 video file
writer = imageio.get_writer('puck_stack.mp4', fps=env.control_freq)
for frame in frames:
    writer.append_data(frame)
writer.close()

print("Video saved successfully! You can now download or view puck_stack.mp4")

[robosuite INFO] Loading controller configuration from: /usr/local/lib/python3.12/dist-packages/robosuite/controllers/config/robots/default_panda.json (composite_controller_factory.py:121)
INFO:robosuite_logs:Loading controller configuration from: /usr/local/lib/python3.12/dist-packages/robosuite/controllers/config/robots/default_panda.json
[robosuite INFO] Loading controller configuration from: /usr/local/lib/python3.12/dist-packages/robosuite/controllers/config/robots/default_panda.json (composite_controller_factory.py:121)
INFO:robosuite_logs:Loading controller configuration from: /usr/local/lib/python3.12/dist-packages/robosuite/controllers/config/robots/default_panda.json


Running headless PuckStack Environment...
Recording random actions to memory...
Saving video to puck_stack.mp4...
Video saved successfully! You can now download or view puck_stack.mp4


In [5]:
from IPython.display import HTML
from base64 import b64encode

# 1. Read the video file and encode it to base64
video_path = 'puck_stack.mp4'
video_file = open(video_path, "rb").read()
video_url = f"data:video/mp4;base64,{b64encode(video_file).decode()}"

# 2. Create the HTML video player
HTML(f"""
<video width="640" height="480" controls>
      <source src="{video_url}" type="video/mp4">
</video>
""")